In [1]:
import pandas as pd          # main data tool (like Excel in Python)
import numpy as np           # math & number operations
import matplotlib.pyplot as plt  # basic charts
import seaborn as sns        # nicer looking charts
import warnings

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)  # show ALL columns
print('All libraries imported successfully!')

All libraries imported successfully!


In [2]:
df = pd.read_csv('D:\Technocolabs\Pickup Five Cities Datasets\pickup_hz.csv')

In [3]:
print('Number of rows   :', df.shape[0])  # how many records
print('Number of columns:', df.shape[1])  # how many fields

Number of rows   : 2130456
Number of columns: 19


In [4]:
df.head()

,order_id,region_id,city,courier_id,accept_time,time_window_start,time_window_end,lng,lat,aoi_id,aoi_type,pickup_time,pickup_gps_time,pickup_gps_lng,pickup_gps_lat,accept_gps_time,accept_gps_lng,accept_gps_lat,ds
0,4965037,1,Hangzhou,6885,07-14 09:21:00,07-14 17:00:00,07-14 19:00:00,120.11976,30.32921,102,14,07-14 17:14:00,07-14 17:14:00,120.11809,30.33748,07-14 09:21:00,120.13181,30.34463,714
1,1309531,1,Hangzhou,13428,07-19 08:05:00,07-19 09:00:00,07-19 11:00:00,120.02190,30.37915,202,1,07-19 08:47:00,NaN,NaN,NaN,NaN,NaN,NaN,719
2,4390255,1,Hangzhou,13428,10-20 11:24:00,10-20 17:00:00,10-20 19:00:00,120.02184,30.37902,202,1,10-20 17:50:00,NaN,NaN,NaN,NaN,NaN,NaN,1020
3,5702636,1,Hangzhou,13428,10-29 12:48:00,10-29 17:02:00,10-29 19:02:00,120.02195,30.37904,202,1,10-29 17:00:00,NaN,NaN,NaN,NaN,NaN,NaN,1029
4,1187003,1,Hangzhou,13428,10-16 07:59:00,10-16 09:00:00,10-16 11:00:00,120.02181,30.37916,202,1,10-16 09:56:00,NaN,NaN,NaN,NaN,NaN,NaN,1016


In [5]:
df.dtypes

order_id               int64
region_id              int64
city                  object
courier_id             int64
accept_time           object
time_window_start     object
time_window_end       object
lng                  float64
lat                  float64
aoi_id                 int64
aoi_type               int64
pickup_time           object
pickup_gps_time       object
pickup_gps_lng       float64
pickup_gps_lat       float64
accept_gps_time       object
accept_gps_lng       float64
accept_gps_lat       float64
ds                     int64
dtype: object

In [6]:
df.isnull().sum()

order_id                   0
region_id                  0
city                       0
courier_id                 0
accept_time                0
time_window_start          0
time_window_end            0
lng                        0
lat                        0
aoi_id                     0
aoi_type                   0
pickup_time                0
pickup_gps_time       747022
pickup_gps_lng        747022
pickup_gps_lat        747022
accept_gps_time      1086734
accept_gps_lng       1086734
accept_gps_lat       1086734
ds                         0
dtype: int64

In [7]:
gps_cols = ['pickup_gps_lng', 'pickup_gps_lat',
            'accept_gps_lng',  'accept_gps_lat']
df[gps_cols] = df[gps_cols].fillna(0.0)

In [8]:
print(df[gps_cols].isnull().sum())

pickup_gps_lng    0
pickup_gps_lat    0
accept_gps_lng    0
accept_gps_lat    0
dtype: int64


In [9]:
# Step 1 — Reload the original file fresh (resets everything)
df = pd.read_csv('D:\Technocolabs\Pickup Five Cities Datasets\pickup_hz.csv')

# Step 2 — Convert datetime columns (format is MM-DD HH:MM:SS, add year manually)
time_cols = ['accept_time', 'time_window_start', 'time_window_end', 'pickup_time']

for col in time_cols:
    df[col] = pd.to_datetime('2023-' + df[col],
                              format='%Y-%m-%d %H:%M:%S',
                              errors='coerce')

# Step 3 — Verify it worked
print(df[time_cols].dtypes)
print()
print(df['accept_time'].head())

accept_time          datetime64[ns]
time_window_start    datetime64[ns]
time_window_end      datetime64[ns]
pickup_time          datetime64[ns]
dtype: object

0   2023-07-14 09:21:00
1   2023-07-19 08:05:00
2   2023-10-20 11:24:00
3   2023-10-29 12:48:00
4   2023-10-16 07:59:00
Name: accept_time, dtype: datetime64[ns]


In [10]:
duplicates = df.duplicated(subset=['order_id'])
print('Number of duplicate rows:', duplicates.sum())

Number of duplicate rows: 0


In [11]:
# Example: How many minutes from accept to pickup?
df['accept_to_pickup_min'] = (
    (df['pickup_time'] - df['accept_time']).dt.total_seconds() / 60
)
print(df['accept_to_pickup_min'].describe().round(1))
# This shows average, min, max pickup time in minutes

count    2130456.0
mean         231.9
std          428.0
min            0.0
25%           68.0
50%          124.0
75%          208.0
max        31795.0
Name: accept_to_pickup_min, dtype: float64


In [12]:
col = 'accept_to_pickup_min'
Q1 = df[col].quantile(0.25)
Q3 = df[col].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = (df[col] < lower) | (df[col] > upper)
print(f'Outlier rows found: {outliers.sum():,}')
print(f'Valid range: {lower:.0f} to {upper:.0f} minutes')

Outlier rows found: 224,840
Valid range: -142 to 418 minutes


In [13]:
df = df[df['accept_to_pickup_min'] >= 0].reset_index(drop=True)
print('Rows after removing negatives:', len(df))

Rows after removing negatives: 2130456


In [14]:
df['accept_to_pickup_min'] = df['accept_to_pickup_min'].clip(upper=upper)
print('Max duration after clipping:', df['accept_to_pickup_min'].max())

Max duration after clipping: 418.0


In [15]:
df['pickup_on_time'] = (
    (df['pickup_time'] >= df['time_window_start']) &
    (df['pickup_time'] <= df['time_window_end'])
)
on_time_pct = df['pickup_on_time'].mean() * 100
print(f'On-time pickup rate: {on_time_pct:.1f}%')

On-time pickup rate: 75.2%


In [16]:
df['accept_hour']  = df['accept_time'].dt.hour    # 0 to 23
df['accept_day']   = df['accept_time'].dt.day     # 1 to 31
df['accept_month'] = df['accept_time'].dt.month   # 1 to 12
print(df[['accept_time', 'accept_hour', 'accept_day', 'accept_month']].head(3))

          accept_time  accept_hour  accept_day  accept_month
0 2023-07-14 09:21:00            9          14             7
1 2023-07-19 08:05:00            8          19             7
2 2023-10-20 11:24:00           11          20            10


In [17]:
def time_of_day(hour):
    if   0 <= hour < 6:   return 'Night'
    elif 6 <= hour < 12:  return 'Morning'
    elif 12 <= hour < 17: return 'Afternoon'
    elif 17 <= hour < 21: return 'Evening'
    else:                  return 'Night'
df['time_of_day'] = df['accept_hour'].apply(time_of_day)
print(df['time_of_day'].value_counts())
print(df['time_of_day'].value_counts().sum())

time_of_day
Morning      1583797
Afternoon     539257
Evening         6773
Night            629
Name: count, dtype: int64
2130456


In [18]:
df['window_duration_min'] = (
    (df['time_window_end'] - df['time_window_start']).dt.total_seconds() / 60
)
print(df['window_duration_min'].value_counts().head())


window_duration_min
120.0    2048654
60.0       55619
899.0       8815
772.0         50
597.0         41
Name: count, dtype: int64


In [19]:
print('Final dataset shape:', df.shape)
print()
print('Missing values left:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print()
print('New columns created:')
new_cols = ['has_pickup_gps', 'has_accept_gps', 'accept_to_pickup_min',
            'pickup_on_time', 'accept_hour', 'accept_day', 'accept_month',
            'time_of_day', 'window_duration_min']
print(new_cols)

Final dataset shape: (2130456, 26)

Missing values left:
pickup_gps_time     747022
pickup_gps_lng      747022
pickup_gps_lat      747022
accept_gps_time    1086734
accept_gps_lng     1086734
accept_gps_lat     1086734
dtype: int64

New columns created:
['has_pickup_gps', 'has_accept_gps', 'accept_to_pickup_min', 'pickup_on_time', 'accept_hour', 'accept_day', 'accept_month', 'time_of_day', 'window_duration_min']


In [20]:
# Check all text columns and their unique values
text_cols = df.select_dtypes(include='object').columns.tolist()
print('Text columns:', text_cols)
print()

for col in text_cols:
    print(f'--- {col} ---')
    print(df[col].value_counts())
    print()
    

Text columns: ['city', 'pickup_gps_time', 'accept_gps_time', 'time_of_day']

--- city ---
city
Hangzhou    2130456
Name: count, dtype: int64

--- pickup_gps_time ---
pickup_gps_time
06-23 09:21:00    45
06-19 11:35:00    43
06-19 10:35:00    43
06-25 09:52:00    42
06-19 16:42:00    42
                  ..
05-28 20:16:00     1
08-27 18:34:00     1
09-02 07:51:00     1
09-27 18:26:00     1
06-25 19:28:00     1
Name: count, Length: 123278, dtype: int64

--- accept_gps_time ---
accept_gps_time
06-21 08:35:00    140
06-21 08:39:00    124
06-07 08:37:00    119
06-25 08:37:00    114
06-20 08:43:00    114
                 ... 
09-03 17:38:00      1
05-15 17:10:00      1
07-30 18:16:00      1
10-01 12:43:00      1
08-09 14:26:00      1
Name: count, Length: 107139, dtype: int64

--- time_of_day ---
time_of_day
Morning      1583797
Afternoon     539257
Evening         6773
Night            629
Name: count, dtype: int64



In [21]:
for col in text_cols:
    df[col] = df[col].str.strip()        # remove spaces before/after
    df[col] = df[col].str.title()        # Capitalize First Letter
    df[col] = df[col].str.replace(r'\s+', ' ', regex=True)  # remove double spaces

print('Text columns cleaned!')
print(df['city'].value_counts())

Text columns cleaned!
city
Hangzhou    2130456
Name: count, dtype: int64


In [22]:
# Cell 12b — Save cleaned data to a new CSV file
df.to_csv('pickup_hz_cleaned.csv', index=False)
print('File saved as: pickup_hz_cleaned.csv')
print(f'Total rows saved: {len(df):,}')

File saved as: pickup_hz_cleaned.csv
Total rows saved: 2,130,456
